In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [10]:
from sklearn.datasets import fetch_openml

# Load Titanic dataset
# "survived" is the target column for classification

titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame.copy()

# Convert target to numeric values for sklearn
# 0 = did not survive, 1 = survived
df['survived'] = df['survived'].astype(int)

# Separate features and target
X = df.drop(columns=['survived'])
y = df['survived']

# Convert categorical/text columns into numeric form
X = pd.get_dummies(X, drop_first=True)

# Fill missing numeric values using the median
numeric_cols = X.select_dtypes(include=['number']).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# Try different values of k and compare accuracy
k_values = [1, 3, 5, 7, 9, 11]
results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results.append((k, acc))
    print(f"k = {k} -> Accuracy: {acc:.4f}")

best_k, best_acc = max(results, key=lambda x: x[1])
print(f"\nBest k: {best_k} with accuracy: {best_acc:.4f}")

# Train final model with the best k
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

k = 1 -> Accuracy: 0.6641
k = 3 -> Accuracy: 0.6031
k = 5 -> Accuracy: 0.5573
k = 7 -> Accuracy: 0.5534
k = 9 -> Accuracy: 0.5496
k = 11 -> Accuracy: 0.5496

Best k: 1 with accuracy: 0.6641

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.93      0.75       144
           1       0.80      0.34      0.48       118

    accuracy                           0.66       262
   macro avg       0.72      0.63      0.61       262
weighted avg       0.71      0.66      0.63       262


Confusion Matrix:
[[134  10]
 [ 78  40]]
